<a href="https://colab.research.google.com/github/renato-avellar/ETL-VENDAS-DESAFIO-3-DPT/blob/main/etl_vendas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [64]:
import duckdb
import pandas as pd
import numpy as np

In [65]:
path = '/content/electronics_sales_raw.csv'
df_bronze = duckdb.query(f"SELECT * FROM read_csv_auto('{path}')").df()
display(df_bronze)

,order_id,customer_id,customer_name,customer_segment,customer_type,first_purchase_date,last_purchase_date,product_id,product_name,category,...,discount_pct,sales_channel,payment_method,sales_rep,region,operating_expenses,cash_balance,debt_balance,monthly_burn,churn_flag
0,10001,C1689,Thiago Alves,Consumer,New,2023-08-26,2024-04-24,P1001,MX Keys S,Electronics,...,0.05,Online,Credit Card,Eduardo,South,239.83,14629.23,3092.03,329.57,0
1,10002,C1002,Juliana Araújo,Consumer,New,2023-02-26,2025-07-15,P1002,Redmi Note 13,Electronics,...,0.15,Online,Credit Card,Diego,North,81.55,8719.54,1189.63,88.62,1
2,10003,C1422,Igor Costa,Consumer,New,2023-03-10,2024-01-01,P1003,iPad mini,Electronics,...,0.15,Online,Cash,Camila,South,244.79,9475.50,3302.27,389.89,0
3,10004,C1308,João Silva,Consumer,New,2023-10-02,2024-01-17,P1004,Galaxy S24,Electronics,...,0.00,Retail,Credit Card,Camila,West,304.91,12381.10,122.53,306.67,0
4,10005,C0826,Vanessa Araújo,Consumer,New,2023-11-04,2024-12-17,P1002,Redmi Note 13,Electronics,...,0.05,Online,Credit Card,Eduardo,Central,24.90,12661.78,2031.60,317.36,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6995,16996,C1164,Ana Rocha,Consumer,Returning,2022-11-25,2025-10-06,P1014,WH-1000XM5,Electronics,...,0.05,Online,Credit Card,Bruno,South,44.92,9181.03,0.00,378.89,0
6996,16997,C1555,Gabriel Silva,Consumer,Returning,2022-10-17,2025-12-10,P1034,iPhone 15 Plus,Electronics,...,0.10,Online,Credit Card,Diego,South,244.44,15611.15,2000.66,427.75,0
6997,16998,C1169,Patrícia Gomes,Consumer,Returning,2023-06-05,2025-07-22,P1027,320K Keyboard,Electronics,...,0.10,Retail,Credit Card,Bruno,South,143.83,17857.85,4046.86,321.57,0
6998,16999,C1374,Ana Souza,Consumer,Returning,2023-06-11,2025-10-28,P1030,Flip 6,Electronics,...,0.00,Online,Debit Card,Camila,East,283.17,16181.14,3545.08,386.07,0


In [66]:
len(df_bronze)

7000

In [67]:
df_silver = df_bronze.copy()
df_silver.head(10)

,order_id,customer_id,customer_name,customer_segment,customer_type,first_purchase_date,last_purchase_date,product_id,product_name,category,...,discount_pct,sales_channel,payment_method,sales_rep,region,operating_expenses,cash_balance,debt_balance,monthly_burn,churn_flag
0,10001,C1689,Thiago Alves,Consumer,New,2023-08-26,2024-04-24,P1001,MX Keys S,Electronics,...,0.05,Online,Credit Card,Eduardo,South,239.83,14629.23,3092.03,329.57,0
1,10002,C1002,Juliana Araújo,Consumer,New,2023-02-26,2025-07-15,P1002,Redmi Note 13,Electronics,...,0.15,Online,Credit Card,Diego,North,81.55,8719.54,1189.63,88.62,1
2,10003,C1422,Igor Costa,Consumer,New,2023-03-10,2024-01-01,P1003,iPad mini,Electronics,...,0.15,Online,Cash,Camila,South,244.79,9475.50,3302.27,389.89,0
3,10004,C1308,João Silva,Consumer,New,2023-10-02,2024-01-17,P1004,Galaxy S24,Electronics,...,0.00,Retail,Credit Card,Camila,West,304.91,12381.10,122.53,306.67,0
4,10005,C0826,Vanessa Araújo,Consumer,New,2023-11-04,2024-12-17,P1002,Redmi Note 13,Electronics,...,0.05,Online,Credit Card,Eduardo,Central,24.90,12661.78,2031.60,317.36,0
5,10006,C0655,Pedro Ribeiro,Consumer,New,2023-07-30,2025-04-29,P1005,Latitude 5440,Electronics,...,0.05,Online,Debit Card,Eduardo,East,304.33,13556.28,5138.21,125.13,0
6,10007,C0779,Mariana Martins,Consumer,New,2023-06-09,2025-05-01,P1006,iPhone 15,Electronics,...,0.00,Retail,Debit Card,Ana,West,128.99,11662.97,1009.29,163.64,0
7,10008,C0962,Lucas Costa,Consumer,New,2022-08-10,2025-12-21,P1007,iPhone 14,Electronics,...,0.05,Online,Credit Card,Ana,Central,148.88,7694.68,86.38,155.27,0
8,10009,C1428,Aline Rocha,Corporate,New,2022-12-17,2024-02-19,P1008,HomePod mini,Electronics,...,0.05,Online,Credit Card,Bruno,North,101.41,10240.91,1890.63,50.37,0
9,10010,C0318,Mateus Ribeiro,Consumer,New,2023-05-16,2025-05-08,P1009,iPad Air,Electronics,...,0.15,Retail,Cash,Diego,North,105.10,8898.65,2903.98,110.74,1


In [68]:

col_category = ['category', 'sub_category', 'brand', 'sales_channel',
               'region', 'payment_method', 'sales_rep', 'customer_type']

# Retorna uma série com a contagem de cada uma
print(df_silver[col_category].nunique())

category           1
sub_category       5
brand             10
sales_channel      2
region             5
payment_method     3
sales_rep          5
customer_type      2
dtype: int64


In [69]:
df_silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Data columns (total 25 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   order_id             7000 non-null   int64         
 1   customer_id          7000 non-null   object        
 2   customer_name        7000 non-null   object        
 3   customer_segment     7000 non-null   object        
 4   customer_type        7000 non-null   object        
 5   first_purchase_date  7000 non-null   datetime64[us]
 6   last_purchase_date   7000 non-null   datetime64[us]
 7   product_id           7000 non-null   object        
 8   product_name         7000 non-null   object        
 9   category             7000 non-null   object        
 10  sub_category         7000 non-null   object        
 11  brand                7000 non-null   object        
 12  order_date           7000 non-null   datetime64[us]
 13  quantity             7000 non-nul

In [70]:
str_cols = ['customer_id', 'customer_name', 'customer_segment', 'customer_type', 'product_id', 'product_name', 'category', 'sub_category', 'brand', 'sales_channel', 'payment_method', 'sales_rep', 'region']

for col in str_cols:
    df_silver[col] = df_silver[col].astype('string').str.strip().str.title()

df_silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Data columns (total 25 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   order_id             7000 non-null   int64         
 1   customer_id          7000 non-null   string        
 2   customer_name        7000 non-null   string        
 3   customer_segment     7000 non-null   string        
 4   customer_type        7000 non-null   string        
 5   first_purchase_date  7000 non-null   datetime64[us]
 6   last_purchase_date   7000 non-null   datetime64[us]
 7   product_id           7000 non-null   string        
 8   product_name         7000 non-null   string        
 9   category             7000 non-null   string        
 10  sub_category         7000 non-null   string        
 11  brand                7000 non-null   string        
 12  order_date           7000 non-null   datetime64[us]
 13  quantity             7000 non-nul

In [71]:
df_silver['customer_id'] = df_silver['customer_id'].str.upper()
df_silver['customer_id'] = df_silver['product_id'].str.upper()

In [72]:
dates_col = ['first_purchase_date', 'last_purchase_date', 'order_date']

for col in dates_col:
  df_silver[col] = pd.to_datetime(df_silver[col])

In [73]:
df_silver = df_silver[(df_silver['quantity'] > 0) & (df_silver['unit_price'] > 0)]
len(df_silver)

7000

In [74]:
df_silver['gross_revenue'] = df_silver['quantity'] * df_silver['unit_price']
df_silver['net_revenue'] = df_silver['gross_revenue'] * (1 - df_silver['discount_pct'])
df_silver

,order_id,customer_id,customer_name,customer_segment,customer_type,first_purchase_date,last_purchase_date,product_id,product_name,category,...,payment_method,sales_rep,region,operating_expenses,cash_balance,debt_balance,monthly_burn,churn_flag,gross_revenue,net_revenue
0,10001,P1001,Thiago Alves,Consumer,New,2023-08-26,2024-04-24,P1001,Mx Keys S,Electronics,...,Credit Card,Eduardo,South,239.83,14629.23,3092.03,329.57,0,439.08,417.1260
1,10002,P1002,Juliana Araújo,Consumer,New,2023-02-26,2025-07-15,P1002,Redmi Note 13,Electronics,...,Credit Card,Diego,North,81.55,8719.54,1189.63,88.62,1,1050.87,893.2395
2,10003,P1003,Igor Costa,Consumer,New,2023-03-10,2024-01-01,P1003,Ipad Mini,Electronics,...,Cash,Camila,South,244.79,9475.50,3302.27,389.89,0,1287.96,1094.7660
3,10004,P1004,João Silva,Consumer,New,2023-10-02,2024-01-17,P1004,Galaxy S24,Electronics,...,Credit Card,Camila,West,304.91,12381.10,122.53,306.67,0,844.50,844.5000
4,10005,P1002,Vanessa Araújo,Consumer,New,2023-11-04,2024-12-17,P1002,Redmi Note 13,Electronics,...,Credit Card,Eduardo,Central,24.90,12661.78,2031.60,317.36,0,332.72,316.0840
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6995,16996,P1014,Ana Rocha,Consumer,Returning,2022-11-25,2025-10-06,P1014,Wh-1000Xm5,Electronics,...,Credit Card,Bruno,South,44.92,9181.03,0.00,378.89,0,647.80,615.4100
6996,16997,P1034,Gabriel Silva,Consumer,Returning,2022-10-17,2025-12-10,P1034,Iphone 15 Plus,Electronics,...,Credit Card,Diego,South,244.44,15611.15,2000.66,427.75,0,1151.16,1036.0440
6997,16998,P1027,Patrícia Gomes,Consumer,Returning,2023-06-05,2025-07-22,P1027,320K Keyboard,Electronics,...,Credit Card,Bruno,South,143.83,17857.85,4046.86,321.57,0,148.08,133.2720
6998,16999,P1030,Ana Souza,Consumer,Returning,2023-06-11,2025-10-28,P1030,Flip 6,Electronics,...,Debit Card,Camila,East,283.17,16181.14,3545.08,386.07,0,375.57,375.5700


In [75]:
df_silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Data columns (total 27 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   order_id             7000 non-null   int64         
 1   customer_id          7000 non-null   string        
 2   customer_name        7000 non-null   string        
 3   customer_segment     7000 non-null   string        
 4   customer_type        7000 non-null   string        
 5   first_purchase_date  7000 non-null   datetime64[us]
 6   last_purchase_date   7000 non-null   datetime64[us]
 7   product_id           7000 non-null   string        
 8   product_name         7000 non-null   string        
 9   category             7000 non-null   string        
 10  sub_category         7000 non-null   string        
 11  brand                7000 non-null   string        
 12  order_date           7000 non-null   datetime64[us]
 13  quantity             7000 non-nul

In [76]:
cogs_map = {
        'Smartphones': 0.80,
        'Tablets': 0.75,
        'Peripherals': 0.50,
        'Laptops': 0.85
    }

df_silver['cogs_pct'] = df_silver['sub_category'].map(cogs_map).fillna(0.7)
df_silver['cost_of_goods_sold'] = df_silver['gross_revenue'] * df_silver['cogs_pct']

df_silver

,order_id,customer_id,customer_name,customer_segment,customer_type,first_purchase_date,last_purchase_date,product_id,product_name,category,...,region,operating_expenses,cash_balance,debt_balance,monthly_burn,churn_flag,gross_revenue,net_revenue,cogs_pct,cost_of_goods_sold
0,10001,P1001,Thiago Alves,Consumer,New,2023-08-26,2024-04-24,P1001,Mx Keys S,Electronics,...,South,239.83,14629.23,3092.03,329.57,0,439.08,417.1260,0.50,219.5400
1,10002,P1002,Juliana Araújo,Consumer,New,2023-02-26,2025-07-15,P1002,Redmi Note 13,Electronics,...,North,81.55,8719.54,1189.63,88.62,1,1050.87,893.2395,0.80,840.6960
2,10003,P1003,Igor Costa,Consumer,New,2023-03-10,2024-01-01,P1003,Ipad Mini,Electronics,...,South,244.79,9475.50,3302.27,389.89,0,1287.96,1094.7660,0.75,965.9700
3,10004,P1004,João Silva,Consumer,New,2023-10-02,2024-01-17,P1004,Galaxy S24,Electronics,...,West,304.91,12381.10,122.53,306.67,0,844.50,844.5000,0.80,675.6000
4,10005,P1002,Vanessa Araújo,Consumer,New,2023-11-04,2024-12-17,P1002,Redmi Note 13,Electronics,...,Central,24.90,12661.78,2031.60,317.36,0,332.72,316.0840,0.80,266.1760
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6995,16996,P1014,Ana Rocha,Consumer,Returning,2022-11-25,2025-10-06,P1014,Wh-1000Xm5,Electronics,...,South,44.92,9181.03,0.00,378.89,0,647.80,615.4100,0.70,453.4600
6996,16997,P1034,Gabriel Silva,Consumer,Returning,2022-10-17,2025-12-10,P1034,Iphone 15 Plus,Electronics,...,South,244.44,15611.15,2000.66,427.75,0,1151.16,1036.0440,0.80,920.9280
6997,16998,P1027,Patrícia Gomes,Consumer,Returning,2023-06-05,2025-07-22,P1027,320K Keyboard,Electronics,...,South,143.83,17857.85,4046.86,321.57,0,148.08,133.2720,0.50,74.0400
6998,16999,P1030,Ana Souza,Consumer,Returning,2023-06-11,2025-10-28,P1030,Flip 6,Electronics,...,East,283.17,16181.14,3545.08,386.07,0,375.57,375.5700,0.70,262.8990


In [77]:
df_silver['gross_profit'] = df_silver['net_revenue'] - df_silver['cost_of_goods_sold']
df_silver

,order_id,customer_id,customer_name,customer_segment,customer_type,first_purchase_date,last_purchase_date,product_id,product_name,category,...,operating_expenses,cash_balance,debt_balance,monthly_burn,churn_flag,gross_revenue,net_revenue,cogs_pct,cost_of_goods_sold,gross_profit
0,10001,P1001,Thiago Alves,Consumer,New,2023-08-26,2024-04-24,P1001,Mx Keys S,Electronics,...,239.83,14629.23,3092.03,329.57,0,439.08,417.1260,0.50,219.5400,197.5860
1,10002,P1002,Juliana Araújo,Consumer,New,2023-02-26,2025-07-15,P1002,Redmi Note 13,Electronics,...,81.55,8719.54,1189.63,88.62,1,1050.87,893.2395,0.80,840.6960,52.5435
2,10003,P1003,Igor Costa,Consumer,New,2023-03-10,2024-01-01,P1003,Ipad Mini,Electronics,...,244.79,9475.50,3302.27,389.89,0,1287.96,1094.7660,0.75,965.9700,128.7960
3,10004,P1004,João Silva,Consumer,New,2023-10-02,2024-01-17,P1004,Galaxy S24,Electronics,...,304.91,12381.10,122.53,306.67,0,844.50,844.5000,0.80,675.6000,168.9000
4,10005,P1002,Vanessa Araújo,Consumer,New,2023-11-04,2024-12-17,P1002,Redmi Note 13,Electronics,...,24.90,12661.78,2031.60,317.36,0,332.72,316.0840,0.80,266.1760,49.9080
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6995,16996,P1014,Ana Rocha,Consumer,Returning,2022-11-25,2025-10-06,P1014,Wh-1000Xm5,Electronics,...,44.92,9181.03,0.00,378.89,0,647.80,615.4100,0.70,453.4600,161.9500
6996,16997,P1034,Gabriel Silva,Consumer,Returning,2022-10-17,2025-12-10,P1034,Iphone 15 Plus,Electronics,...,244.44,15611.15,2000.66,427.75,0,1151.16,1036.0440,0.80,920.9280,115.1160
6997,16998,P1027,Patrícia Gomes,Consumer,Returning,2023-06-05,2025-07-22,P1027,320K Keyboard,Electronics,...,143.83,17857.85,4046.86,321.57,0,148.08,133.2720,0.50,74.0400,59.2320
6998,16999,P1030,Ana Souza,Consumer,Returning,2023-06-11,2025-10-28,P1030,Flip 6,Electronics,...,283.17,16181.14,3545.08,386.07,0,375.57,375.5700,0.70,262.8990,112.6710


In [79]:
df_silver['order_year'] = df_silver['order_date'].dt.year
df_silver['order_month'] = df_silver['order_date'].dt.month
df_silver['order_quarter'] = df_silver['order_date'].dt.quarter
df_silver['order_month_year'] = df_silver['order_date'].dt.strftime('%Y-%m')

df_silver

,order_id,customer_id,customer_name,customer_segment,customer_type,first_purchase_date,last_purchase_date,product_id,product_name,category,...,churn_flag,gross_revenue,net_revenue,cogs_pct,cost_of_goods_sold,gross_profit,order_year,order_month,order_quarter,order_month_year
0,10001,P1001,Thiago Alves,Consumer,New,2023-08-26,2024-04-24,P1001,Mx Keys S,Electronics,...,0,439.08,417.1260,0.50,219.5400,197.5860,2024,4,2,2024-04
1,10002,P1002,Juliana Araújo,Consumer,New,2023-02-26,2025-07-15,P1002,Redmi Note 13,Electronics,...,1,1050.87,893.2395,0.80,840.6960,52.5435,2025,7,3,2025-07
2,10003,P1003,Igor Costa,Consumer,New,2023-03-10,2024-01-01,P1003,Ipad Mini,Electronics,...,0,1287.96,1094.7660,0.75,965.9700,128.7960,2024,1,1,2024-01
3,10004,P1004,João Silva,Consumer,New,2023-10-02,2024-01-17,P1004,Galaxy S24,Electronics,...,0,844.50,844.5000,0.80,675.6000,168.9000,2024,1,1,2024-01
4,10005,P1002,Vanessa Araújo,Consumer,New,2023-11-04,2024-12-17,P1002,Redmi Note 13,Electronics,...,0,332.72,316.0840,0.80,266.1760,49.9080,2024,12,4,2024-12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6995,16996,P1014,Ana Rocha,Consumer,Returning,2022-11-25,2025-10-06,P1014,Wh-1000Xm5,Electronics,...,0,647.80,615.4100,0.70,453.4600,161.9500,2025,2,1,2025-02
6996,16997,P1034,Gabriel Silva,Consumer,Returning,2022-10-17,2025-12-10,P1034,Iphone 15 Plus,Electronics,...,0,1151.16,1036.0440,0.80,920.9280,115.1160,2025,6,2,2025-06
6997,16998,P1027,Patrícia Gomes,Consumer,Returning,2023-06-05,2025-07-22,P1027,320K Keyboard,Electronics,...,0,148.08,133.2720,0.50,74.0400,59.2320,2025,7,3,2025-07
6998,16999,P1030,Ana Souza,Consumer,Returning,2023-06-11,2025-10-28,P1030,Flip 6,Electronics,...,0,375.57,375.5700,0.70,262.8990,112.6710,2024,1,1,2024-01


In [80]:
print("--- Lucro Bruto por Região ---")
print(df_silver.groupby('region')['gross_profit'].sum().sort_values(ascending=False))

print("\n--- Top 5 Categorias por Receita Líquida ---")
print(df_silver.groupby('category')['net_revenue'].sum().sort_values(ascending=False).head(5))

print("\n--- Margem Média por Subcategoria (%) ---")
df_silver['margin_pct'] = (df_silver['gross_profit'] / df_silver['net_revenue']) * 100
print(df_silver.groupby('sub_category')['margin_pct'].mean().sort_values(ascending=False))

--- Lucro Bruto por Região ---
region
North      168172.7895
West       157041.1345
Central    155242.4065
South      153926.9290
East       144380.6450
Name: gross_profit, dtype: float64

--- Top 5 Categorias por Receita Líquida ---
category
Electronics    5472966.183
Name: net_revenue, dtype: float64

--- Margem Média por Subcategoria (%) ---
sub_category
Peripherals    45.182190
Audio          23.181933
Tablets        17.684724
Smartphones    12.745249
Laptops         7.309954
Name: margin_pct, dtype: float64


In [81]:
df_silver.to_csv('electronics_sales_silver.csv', index=False, encoding='utf-8')